In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

# Main project folder name
project_name = "ISL_GAN_FewShot_Project"

# Base path inside Google Drive
base_path = f"/content/drive/MyDrive/{project_name}"

# Folder structure
folders = [
    base_path,
    f"{base_path}/data",
    f"{base_path}/data/raw",
    f"{base_path}/data/processed",
    f"{base_path}/data/train",
    f"{base_path}/data/val",
    f"{base_path}/data/test",
    f"{base_path}/data/synthetic",
    f"{base_path}/data/landmarks",

    f"{base_path}/models",
    f"{base_path}/models/baseline",
    f"{base_path}/models/gan",
    f"{base_path}/models/fewshot",
    f"{base_path}/models/checkpoints",

    f"{base_path}/logs",
    f"{base_path}/plots",
    f"{base_path}/results",
    f"{base_path}/results/accuracy",
    f"{base_path}/results/loss",
    f"{base_path}/results/confusion_matrix",

    f"{base_path}/notebooks",
    f"{base_path}/src",
    f"{base_path}/app",
    f"{base_path}/docs",
    f"{base_path}/docs/reports",
    f"{base_path}/docs/presentation"
]

# Create folders
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folder structure created successfully at:")
print(base_path)

Project folder structure created successfully at:
/content/drive/MyDrive/ISL_GAN_FewShot_Project


In [ ]:
import os
import shutil

# Create .kaggle folder
os.makedirs("/root/.kaggle", exist_ok=True)

# Move kaggle.json to correct place
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")

# Set permissions
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle API configured successfully.")

Kaggle API configured successfully.


In [ ]:
# Your project raw data folder
raw_data_path = "/content/drive/MyDrive/ISL_GAN_FewShot_Project/data/raw"

# Create folder if not exists
os.makedirs(raw_data_path, exist_ok=True)

# Move into raw data folder
%cd $raw_data_path

# Download dataset
!kaggle datasets download -d prathumarikeri/indian-sign-language-isl

/content/drive/MyDrive/ISL_GAN_FewShot_Project/data/raw
Dataset URL: https://www.kaggle.com/datasets/prathumarikeri/indian-sign-language-isl
License(s): CC-BY-SA-4.0
100% 281M/281M [00:03<00:00, 83.0MB/s]



In [ ]:
import zipfile

zip_path = f"{raw_data_path}/indian-sign-language-isl.zip"
extract_path = f"{raw_data_path}/indian-sign-language-isl"

# Extract
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")
print("Extracted to:", extract_path)

Dataset extracted successfully.
Extracted to: /content/drive/MyDrive/ISL_GAN_FewShot_Project/data/raw/indian-sign-language-isl


In [ ]:
import os
raw_data_path = "/content/drive/MyDrive/ISL_GAN_FewShot_Project/data/raw"
extract_path = f"{raw_data_path}/indian-sign-language-isl"
dataset_root = extract_path

print("Folders inside dataset:\n")
for item in os.listdir(dataset_root):
    print(item)

Folders inside dataset:

Indian


In [ ]:
from collections import defaultdict

class_counts = defaultdict(int)

for root, dirs, files in os.walk(dataset_root):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            class_name = os.path.basename(root)
            class_counts[class_name] += 1

print(f"Total Classes: {len(class_counts)}\n")

for cls, count in sorted(class_counts.items()):
    print(f"{cls}: {count} images")

Total Classes: 35

1: 1200 images
2: 1200 images
3: 1200 images
4: 1200 images
5: 1200 images
6: 1200 images
7: 1200 images
8: 1200 images
9: 1200 images
A: 1200 images
B: 1200 images
C: 1447 images
D: 1200 images
E: 1200 images
F: 1200 images
G: 1200 images
H: 1200 images
I: 1379 images
J: 1200 images
K: 1200 images
L: 1200 images
M: 1200 images
N: 1200 images
O: 1429 images
P: 1200 images
Q: 1200 images
R: 1200 images
S: 1200 images
T: 1200 images
U: 1200 images
V: 1290 images
W: 1200 images
X: 1200 images
Y: 1200 images
Z: 1200 images


In [ ]:
import os
from pathlib import Path

project_path = Path("/content/drive/MyDrive/ISL_GAN_FewShot_Project")
raw_path = project_path / "data" / "raw"
processed_path = project_path / "data" / "processed"
fewshot_path = processed_path / "fewshot"

# Create required folders
for p in [
    processed_path,
    fewshot_path,
    fewshot_path / "train",
    fewshot_path / "val",
    fewshot_path / "test",
    project_path / "manifests",
]:
    p.mkdir(parents=True, exist_ok=True)

print("Paths ready.")
print("Project:", project_path)

Paths ready.
Project: /content/drive/MyDrive/ISL_GAN_FewShot_Project


In [ ]:
from pathlib import Path

def find_dataset_root(search_root: Path):
    """
    Finds a folder inside search_root that contains class subfolders with images.
    """
    for root, dirs, files in os.walk(search_root):
        root_path = Path(root)
        image_files = list(root_path.glob("*.jpg")) + list(root_path.glob("*.jpeg")) + list(root_path.glob("*.png"))
        subdirs = [d for d in root_path.iterdir() if d.is_dir()]

        # dataset root should have subfolders (classes), not just images
        if subdirs:
            # check if at least one subdir contains images
            for sd in subdirs:
                imgs = list(sd.glob("*.jpg")) + list(sd.glob("*.jpeg")) + list(sd.glob("*.png"))
                if imgs:
                    return root_path
    return None

dataset_root = find_dataset_root(raw_path)

if dataset_root is None:
    raise FileNotFoundError("Could not find the extracted dataset folder inside data/raw. Please check extraction.")

print("Dataset root found at:")
print(dataset_root)

Dataset root found at:
/content/drive/MyDrive/ISL_GAN_FewShot_Project/data/raw/indian-sign-language-isl/Indian


In [ ]:
from collections import defaultdict

def get_class_counts(dataset_root: Path):
    counts = {}
    for class_dir in sorted([d for d in dataset_root.iterdir() if d.is_dir() and not d.name.startswith(".")]):
        images = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.jpeg")) + list(class_dir.glob("*.png"))
        counts[class_dir.name] = len(images)
    return counts

class_counts = get_class_counts(dataset_root)

print("Total classes:", len(class_counts))
for cls, count in list(class_counts.items())[:20]:
    print(f"{cls}: {count}")

Total classes: 35
1: 1200
2: 1200
3: 1200
4: 1200
5: 1200
6: 1200
7: 1200
8: 1200
9: 1200
A: 1200
B: 1200
C: 1447
D: 1200
E: 1200
F: 1200
G: 1200
H: 1200
I: 1379
J: 1200
K: 1200


In [ ]:
import random
import shutil
import pandas as pd

random.seed(42)

K_SHOT = 5          # change to 1, 3, 5, 10 as needed
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Output directories
train_dir = fewshot_path / "train"
val_dir = fewshot_path / "val"
test_dir = fewshot_path / "test"

def clear_folder(folder: Path):
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

# Clear old splits if any
for folder in [train_dir, val_dir, test_dir]:
    clear_folder(folder)

manifest_rows = []

for class_dir in sorted([d for d in dataset_root.iterdir() if d.is_dir() and not d.name.startswith(".")]):
    cls = class_dir.name
    images = [p for p in class_dir.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    images = sorted(images)
    random.shuffle(images)

    if len(images) < K_SHOT + 3:
        print(f"Skipping {cls} because it has too few images: {len(images)}")
        continue

    # few-shot train selection
    train_imgs = images[:K_SHOT]
    remaining = images[K_SHOT:]

    # split remaining into val/test
    n_remaining = len(remaining)
    n_val = max(1, int(n_remaining * VAL_RATIO))
    n_test = max(1, int(n_remaining * TEST_RATIO))

    # adjust if too many
    while n_val + n_test > n_remaining:
        if n_val > n_test and n_val > 1:
            n_val -= 1
        elif n_test > 1:
            n_test -= 1
        else:
            break

    val_imgs = remaining[:n_val]
    test_imgs = remaining[n_val:n_val + n_test]

    # create class folders
    for split_dir in [train_dir, val_dir, test_dir]:
        (split_dir / cls).mkdir(parents=True, exist_ok=True)

    # copy files
    for img in train_imgs:
        dest = train_dir / cls / img.name
        shutil.copy2(img, dest)
        manifest_rows.append([str(img), str(dest), cls, "train"])

    for img in val_imgs:
        dest = val_dir / cls / img.name
        shutil.copy2(img, dest)
        manifest_rows.append([str(img), str(dest), cls, "val"])

    for img in test_imgs:
        dest = test_dir / cls / img.name
        shutil.copy2(img, dest)
        manifest_rows.append([str(img), str(dest), cls, "test"])

    print(f"{cls}: train={len(train_imgs)}, val={len(val_imgs)}, test={len(test_imgs)}")

print("Few-shot split created successfully.")

1: train=5, val=179, test=179
2: train=5, val=179, test=179
3: train=5, val=179, test=179
4: train=5, val=179, test=179
5: train=5, val=179, test=179
6: train=5, val=179, test=179
7: train=5, val=179, test=179
8: train=5, val=179, test=179
9: train=5, val=179, test=179
A: train=5, val=179, test=179
B: train=5, val=179, test=179
C: train=5, val=216, test=216
D: train=5, val=179, test=179
E: train=5, val=179, test=179
F: train=5, val=179, test=179
G: train=5, val=179, test=179
H: train=5, val=179, test=179
I: train=5, val=206, test=206
J: train=5, val=179, test=179
K: train=5, val=179, test=179
L: train=5, val=179, test=179
M: train=5, val=179, test=179
N: train=5, val=179, test=179
O: train=5, val=213, test=213
P: train=5, val=179, test=179
Q: train=5, val=179, test=179
R: train=5, val=179, test=179
S: train=5, val=179, test=179
T: train=5, val=179, test=179
U: train=5, val=179, test=179
V: train=5, val=192, test=192
W: train=5, val=179, test=179
X: train=5, val=179, test=179
Y: train=5

In [ ]:
manifest_df = pd.DataFrame(manifest_rows, columns=["source_path", "copied_path", "class_name", "split"])
manifest_file = project_path / "manifests" / f"fewshot_manifest_k{K_SHOT}.csv"
manifest_df.to_csv(manifest_file, index=False)

print("Manifest saved to:")
print(manifest_file)
print(manifest_df.head())

Manifest saved to:
/content/drive/MyDrive/ISL_GAN_FewShot_Project/manifests/fewshot_manifest_k5.csv
                                         source_path  \
0  /content/drive/MyDrive/ISL_GAN_FewShot_Project...   
1  /content/drive/MyDrive/ISL_GAN_FewShot_Project...   
2  /content/drive/MyDrive/ISL_GAN_FewShot_Project...   
3  /content/drive/MyDrive/ISL_GAN_FewShot_Project...   
4  /content/drive/MyDrive/ISL_GAN_FewShot_Project...   

                                         copied_path class_name  split  
0  /content/drive/MyDrive/ISL_GAN_FewShot_Project...          1  train  
1  /content/drive/MyDrive/ISL_GAN_FewShot_Project...          1  train  
2  /content/drive/MyDrive/ISL_GAN_FewShot_Project...          1  train  
3  /content/drive/MyDrive/ISL_GAN_FewShot_Project...          1  train  
4  /content/drive/MyDrive/ISL_GAN_FewShot_Project...          1  train  
